# CinaMatrix Supabase Import Verification
Verify all 15 CSV files successfully imported to Supabase database with correct schemas and data

## 1. Connect to Supabase and Load Libraries

In [ ]:
import pandas as pd
from supabase import create_client, Client
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

SUPABASE_URL = os.getenv("VITE_SUPABASE_URL")
SUPABASE_KEY = os.getenv("VITE_SUPABASE_ANON_KEY")

# Initialize Supabase client
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print(f"✅ Connected to Supabase: {SUPABASE_URL}")
print(f"✅ Environment variables loaded successfully")

## 2. Verify Table Creation and Row Counts

In [ ]:
# Define all 15 table names
tables = [
    "new_releases",
    "top_rated",
    "reviews",
    "movies_series",
    "awards",
    "movies_by_year",
    "actors",
    "mood_recommendations",
    "recommendation_pool",
    "cinema_galaxy",
    "cinema_world",
    "egypt_cinemas",
    "genre_frequency",
    "genre_cooccurrence",
    "point_cloud"
]

# Check row counts for each table
row_counts = []
for table in tables:
    try:
        response = supabase.table(table).select("*", count="exact").execute()
        row_count = response.count if hasattr(response, 'count') else len(response.data)
        row_counts.append({
            "Table": table,
            "Status": "✅ EXISTS",
            "Row Count": row_count
        })
        print(f"✅ {table}: {row_count} rows")
    except Exception as e:
        row_counts.append({
            "Table": table,
            "Status": "❌ ERROR",
            "Row Count": 0
        })
        print(f"❌ {table}: {str(e)}")

# Display summary
df_counts = pd.DataFrame(row_counts)
print("\n" + "="*50)
print("TABLE VERIFICATION SUMMARY")
print("="*50)
print(df_counts.to_string(index=False))

## 3. Sample Data from Key Tables

In [ ]:
# Show sample data from key tables
key_tables = ["new_releases", "movies_series", "actors", "cinema_world", "genre_frequency"]

for table in key_tables:
    try:
        response = supabase.table(table).select("*").limit(2).execute()
        if response.data:
            df = pd.DataFrame(response.data)
            print(f"\n{'='*80}")
            print(f"📊 {table.upper()} - First 2 rows:")
            print(f"{'='*80}")
            print(df.to_string())
            print(f"\nColumns: {list(df.columns)}")
        else:
            print(f"\n⚠️  {table} has no data")
    except Exception as e:
        print(f"\n❌ Error reading {table}: {str(e)}")

## 4. Data Validation Summary

In [ ]:
print("\n" + "="*80)
print("🎬 CINEMATRIX SUPABASE MIGRATION SUMMARY")
print("="*80)

total_tables = len(df_counts)
existing_tables = len(df_counts[df_counts['Status'] == '✅ EXISTS'])
total_rows = df_counts['Row Count'].sum()

print(f"\n📋 Tables Created: {existing_tables}/{total_tables}")
print(f"📊 Total Rows Imported: {total_rows:,}")
print(f"\n{'Table':<25} {'Rows':<10} {'Status':<10}")
print("-" * 50)

for _, row in df_counts.iterrows():
    status_emoji = "✅" if row['Status'] == '✅ EXISTS' else "❌"
    print(f"{row['Table']:<25} {row['Row Count']:<10} {status_emoji} {row['Status']:<10}")

print("\n" + "="*80)
if existing_tables == total_tables and total_rows > 0:
    print("✨ SUCCESS! All tables created and data imported to Supabase!")
    print("🚀 Ready for frontend testing!")
else:
    print("⚠️  ISSUES DETECTED - Check table status above")
print("="*80)

## 5. Next Steps: Test Frontend

✅ **If all tables show row counts above:**
1. Open terminal in this project folder
2. Run: `python -m http.server 8080`
3. Open browser: **http://localhost:8080**
4. Click on any HTML file (index.html, cinema_galaxy.html, etc.)
5. Open Developer Console (F12) to check for errors
6. Verify visualizations load with data from Supabase

✅ **Frontend expects DATA object with these properties:**
- `new_releases`, `top_rated`, `reviews`, `movies_series`, `awards`
- `movies_by_year`, `actors`, `mood_recommendations`, `recommendation_pool`
- `cinema_galaxy`, `cinema_world`, `egypt_cinemas`, `genre_frequency`
- `genre_cooccurrence`, `point_cloud`